## Hi! Here is the new playground, go ahead and explore the dataset and find out as many insights as you can before jumping to the ML part! 

In [ ]:
import pandas as pd

# Load dataset
df = pd.read_csv('./dataset/housing.csv')
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

##### I identified 207 missing values in the total_bedrooms column. Since each row in this dataset represents a neighborhood rather than a single house, these missing values signify errors or omissions, not a literal zero (a neighborhood of only studios). Replacing these missing values with 0 is dangerous because the model will learn incorrect patterns, such as a large population living with no bedrooms. Because 207 represents a small percentage of our dataset, the first approach that comes to mind is to impute the missing values using the median of the total_bedrooms column. That will be solved when we split the data on train/test to prevent data leakage

##### After analyzing the problematic rows, I observed that the other columns have normal values compared to the rest of the dataset. I based my decision on the median house value, the difference is insignificant, suggesting that the missing values are likely the result of a data collection mistake. Because the number of missing values is small(5%), the missing values are completly random ,probably a mistake in data collection ,and is a numeric type I will use the median value of total_bedrooms to fill in the missing values. This approach is preferable to dropping rows, as it preserves the dataset's size and avoids potential bias from removing data points.

In [ ]:
df_cleaned=df.copy()

In [ ]:
df_cleaned.describe()

##### Now that I observed the dataset problems, I will continue with the analysis. At first I will explore some vizualizations and relations using the "explore-pricing.py" script.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.scatterplot(x='median_income', y='median_house_value', data=df_cleaned, alpha=0.5, hue='ocean_proximity', palette='viridis', legend='full')
plt.title('Median Income vs Median House Value colored by Ocean Proximity')
plt.xlabel('Median Income')
plt.ylabel('Median House Value')
plt.show()

##### I consider this scatter plot to be very relevant because it shows a clear positive correlation between median income and median house value. The diagram shows that as median income increases, the median house value also tends to increase. Additionally, the color coding by ocean proximity reveals that neighborhoods closer to the ocean tend to have higher house values, which is an important insight for our regression model. This suggests that both median income and ocean proximity are significant factors in determining house prices in this dataset. As we can see there is a roof on the datas , the median maximum value is at ~500000 fact that is caused maybe by data collection. I will remove this outliers to analyze the dataset correctly.

In [ ]:
df_cleaned = df_cleaned[df_cleaned['median_house_value'] < 500001].copy()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.scatterplot(x='median_income', y='median_house_value', data=df_cleaned, alpha=0.5, hue='ocean_proximity', palette='viridis', legend='full')
plt.title('Median Income vs Median House Value colored by Ocean Proximity')
plt.xlabel('Median Income')
plt.ylabel('Median House Value')
plt.show()

##### As we can see I eliminated the 'wrong' outliers to analyze a cleaner dataset for better results

In [ ]:
df_cleaned['rooms_per_household'] = df_cleaned['total_rooms'] / df_cleaned['households']
df_cleaned.head()

In [ ]:
df_cleaned['bedrooms_per_rooms']=df_cleaned['total_bedrooms'] / df_cleaned['total_rooms']
df_cleaned

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.scatterplot(x='rooms_per_household', y='median_house_value', data=df_cleaned, alpha=0.5, hue='ocean_proximity', palette='viridis', legend='full')
plt.title('Rooms per Household vs Median House Value colored by Ocean Proximity')
plt.xlabel('Rooms per Household')
plt.ylabel('Median House Value')
plt.show()

##### The scatter plot of rooms per household versus median house value, colored by ocean proximity is not as relevant as the previous one. The relationship between rooms per household and median house value is not as clear or strong as the relationship between median income and median house value. While there may be some correlation, it is not as pronounced, and the data points are more scattered. Additionally, the color coding by ocean proximity does not reveal a significant pattern in this case. Therefore, I would consider this scatter plot to be less relevant for our regression model compared to the previous one.


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.scatterplot(x='bedrooms_per_rooms', y='median_house_value', data=df_cleaned, alpha=0.5, hue='ocean_proximity', palette='viridis', legend='full')
plt.title('Bedrooms per rooms vs Median House Value colored by Ocean Proximity')
plt.xlabel('Bedroooms per rooms')
plt.ylabel('Median House Value')
plt.show()

##### The new feature "bedrooms_per_rooms" seems to be relevant for the median house value but this scatter plot is not the best for visualizing his impact on the desired feature. So I will continue the analyze of this feature to decide his importance.

In [ ]:
corr = df_cleaned.corr(numeric_only=True)

price_corr = corr['median_house_value'].sort_values(ascending=False)

price_corr = price_corr.drop('median_house_value')

plt.figure(figsize=(8, 6))
sns.barplot(x=price_corr.values, y=price_corr.index, palette='viridis')
plt.title('Correlation of Features with Median House Value')
plt.xlabel('Feature')
plt.ylabel('Correlation')
plt.xlabel('Correlation coefficient')
plt.show()

In [ ]:
numeric_cols = ['median_house_value', 'rooms_per_household', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income','housing_median_age', 'longitude', 'latitude','bedrooms_per_rooms']

plt.figure(figsize=(8, 6))
sns.heatmap(df_cleaned[numeric_cols].corr(), annot=True, cmap='coolwarm', center=0, fmt=".2f")
plt.title('Correlation Heatmap of Numeric Features')
plt.show()

##### The heatmap diagram shows the correlation between different features and median house value. The features with the highest positive correlation with median house value are median income, rooms per household, and total rooms. This suggests that neighborhoods with higher median incomes, more rooms per household, and more total rooms tend to have higher house values. On the other hand, features like population and households show a negative correlation with median house value, indicating that neighborhoods with larger populations and more households tend to have lower house values. Overall, this heatmap provides valuable insights into the relationships between different features and median house value, which can inform our future model.

In [ ]:


sns.barplot(x='ocean_proximity', y='median_house_value', data=df_cleaned, hue='ocean_proximity',
    palette='viridis',
    errorbar=None,
    legend=False )
plt.title('Median House Value by Ocean Proximity')
plt.xticks(rotation=45)
plt.show()

##### The diagram shows that the median house value varies significantly based on ocean proximity. Island neighborhoods have the highest median house values, followed by neighborhoods that are near the ocean and the bay. In contrast, neighborhoods that are inland have lower median house values. This suggests that ocean proximity is an important factor in determining house prices in this dataset.

In [ ]:

df_encoded_ocean = pd.get_dummies(df_cleaned[['ocean_proximity', 'median_house_value']], drop_first=False)

corr_ocean = df_encoded_ocean.corr()

corr_with_price = corr_ocean[['median_house_value']].drop('median_house_value')

import seaborn as sns
import matplotlib.pyplot as plt

sns.heatmap(corr_with_price, annot=True, cmap='coolwarm', center=0)
plt.title('Correlation: Ocean Proximity vs Median House Value')
plt.show()

##### The heatmap shows the correlation between ocean proximity and median house value. The diagram indicates how the value evolves based on the proximity to the ocean. The correlation values suggest that neighborhoods that are closer to the ocean tend to have higher median house values, while those that are inland have lower median house values. This reinforces the earlier observation that ocean proximity is a significant factor in determining house prices in this dataset.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.scatterplot(x='population', y='households', data=df_cleaned, alpha=0.5, hue='ocean_proximity', palette='viridis', legend='full')
plt.title('Population vs Households colored by Ocean Proximity')
plt.xlabel('Population')
plt.ylabel('Households')
plt.show()

##### The diagram shows the relationship between population and households colored by ocean proximity. The plot indicates that if the population increases in neighborhood , the number of households also tends to increase, which is expected. The color coding by ocean proximity does not reveal a significant pattern in this case. Overall, this scatter plot provides insights into the relationship between population and households, but it may not be as relevant for our regression model compared to other features.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.scatterplot(x='households', y='total_rooms', data=df_cleaned, alpha=0.5, hue='ocean_proximity', palette='viridis', legend='full')
plt.title('Households vs Total Rooms colored by Ocean Proximity')
plt.xlabel('Households')
plt.ylabel('Total Rooms')
plt.show()

##### The diagram shows the relationship between households and total rooms colored by ocean proximity. The plot indicates that as the number of households increases, the total number of rooms also tends to increase. The same scenario is observed in the previous diagram.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import contextily as ctx
plt.figure(figsize=(10, 8))
ans=sns.scatterplot(x='longitude', y='latitude', data=df_cleaned, alpha=0.5, hue='median_house_value', palette='viridis', s=15)
big_cities = {
    'Los Angeles': (-118.24, 34.05),
    'San Francisco': (-122.41, 37.77),
    'San Diego': (-117.16, 32.71),
    'San Jose': (-121.88, 37.33)
}

for city, (lon, lat) in big_cities.items():
    plt.scatter(lon, lat, color='red',s=150,edgecolors='black', label=city, alpha=0.7)

plt.title('Map of the city')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

ctx.add_basemap(ans,crs='EPSG:4326',source=ctx.providers.OpenStreetMap.Mapnik)
plt.show()

##### This diagram shows the geographical distribution of neighborhoods based on their latitude and longitude colored by median house value. The plot shows the big cities nearby the neighborhoods, which are Los Angeles, San Francisco and San Diego (the big red dots on the map). From what I can see, in the proximity of the big cities around the ocean , prices tend to be higher but I have to do a more detalied analysis to confirm this. Overall, this scatter plot provides insights into the geographical distribution of neighborhoods and their median house values, which can inform our future model. And we can see that there are more neighborhoods in the proximity of the big cities.

In [ ]:
import numpy as np
distances = pd.DataFrame({
    'Los Angeles': np.sqrt((df_cleaned['longitude'] - -118.24)**2 + (df_cleaned['latitude'] - 34.05)**2),
    'San Francisco': np.sqrt((df_cleaned['longitude'] - -122.41)**2 + (df_cleaned['latitude'] - 37.77)**2),
    'San Diego': np.sqrt((df_cleaned['longitude'] - -117.16)**2 + (df_cleaned['latitude'] - 32.71)**2),
    'San Jose': np.sqrt((df_cleaned['longitude'] - -121.88)**2 + (df_cleaned['latitude'] - 37.33)**2)
    # Here I used the basic distance fromula to calculate the distance from the neighborhoods to big cities even if it's not the best approach because the earth is not flat but for this analysis I think it's enough
})

df_cleaned['closest_cities']= distances.idxmin(axis=1)


In [ ]:
df_cleaned

In [ ]:


sns.barplot(x='closest_cities', y='median_house_value', data=df_cleaned, hue='ocean_proximity',
    palette='viridis',
    errorbar=None,
    legend=True )
plt.title('Median House Value by Closest City and Ocean Proximity')
plt.xticks(rotation=45)
plt.show()

##### As we can see in diagram , the median house value varies based on the closest city and ocean proximity. In my opinion the closest city is a relevant factor in determining house prices , especially when combined with the ocean proximity.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.scatterplot(x='housing_median_age', y='median_house_value', data=df_cleaned, alpha=0.5, hue='ocean_proximity', palette='viridis', legend='full')
plt.title('Housing median Age vs Median House value colored by Ocean Proximity')
plt.xlabel('Median Age')
plt.ylabel('Median House Value')
plt.show()

##### The diagram shows the median house evolution based on the housing age. The diagram presents very well that every age group has cheap houses , medium value houses and expensive ones. The coloring by ocean proximity shows once again that the house position is very important for the price evolution. The diagram shows us another issue, that there is a lot of outliers on the 50+ age, that may be a data collection mistake that can influence our models.

In [ ]:
df_cleaned = df_cleaned[df_cleaned['housing_median_age'] < 52].copy()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.scatterplot(x='housing_median_age', y='median_house_value', data=df_cleaned, alpha=0.5, hue='ocean_proximity', palette='viridis', legend='full')
plt.title('Housing median Age vs Median House value colored by Ocean Proximity')
plt.xlabel('Median Age')
plt.ylabel('Median House Value')
plt.show()

In [ ]:
df_encoded = pd.get_dummies(df_cleaned, columns=['ocean_proximity', 'closest_cities'], drop_first=True)

bool_cols = df_encoded.select_dtypes(include=['bool']).columns
df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)

In [ ]:
df_encoded

##### Now that I have my dataset completed and corrected I can start with the ML part. But at first I will solve the NULL values problems from the columns : 'total_bedrooms' and 'bedrooms_per_room'

In [ ]:
from sklearn.model_selection import train_test_split
X = df_encoded.drop(columns=['median_house_value', 'total_rooms', 'total_bedrooms'])
y = df_encoded['median_house_value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
median_bed_per_room = X_train['bedrooms_per_rooms'].median()

X_train['bedrooms_per_rooms'] = X_train['bedrooms_per_rooms'].fillna(median_bed_per_room)
X_test['bedrooms_per_rooms'] = X_test['bedrooms_per_rooms'].fillna(median_bed_per_room)



1. RandomForestRegressor

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

rf=RandomForestRegressor(random_state=42)

param_grid= {
    'n_estimators': [100,150,200],
    'max_depth':[10, 20 , None]
}
cv_model=GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, n_jobs=-1, verbose=2)
cv_model.fit(X_train, y_train)


In [ ]:
best_rf = cv_model.best_estimator_
print("best parameters combination:", cv_model.best_params_)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y_pred = best_rf.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.3f}")

- Based on the performances , the model is acceptable having a: MAE=~29000 , RMSE=~43506 and $R^2$=~80%
- Considering the fact that I used GridSearchCV the problem is with the datas and not the model.
- For improvements I consider adding some new features that can be relevant

In [ ]:
importances = best_rf.feature_importances_
indices = np.argsort(importances)[::-1]
features = X.columns

plt.figure(figsize=(10,6))
plt.title("Feature Importances")
plt.bar(range(len(importances)), importances[indices], align='center')
plt.xticks(range(len(importances)), features[indices], rotation=90)
plt.tight_layout()
plt.show()


- Analyzing the feature importance diagram we can see that how I predicted median_income is a very important feature.
- We can see that some features added by me like ocean_proximity_Island or other ex. are near 0 because of the latitude and longitude weight, the model knows the location already based on these 2 features.

In [ ]:
plt.scatter(y_test, y_pred, alpha=0.3)
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Random Forest: Actual vs Predicted Prices")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.show()

- The regression line diagram shows that the model predicts well the normal values but it has some problems on high value houses.
- To improve the line fit as I said I can work on the features and maybe on the data preprocessing.

2. Linear Regression

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler=StandardScaler()

X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)



In [ ]:
from sklearn.linear_model import LinearRegression
lin_reg=LinearRegression()
lin_reg.fit(X_train_scaled, y_train)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y_pred = lin_reg.predict(X_test_scaled)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.3f}")

In [ ]:
plt.scatter(y_test, y_pred, alpha=0.3)
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Linear Regression: Actual vs Predicted Prices")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.show()

- As expected the linear regression is not as good as the RF model.
- The house pricing evolution is hard to predict using a linear mathematical model.
- The diagram shows that the model performs good on the normal values but on the high value houses it fails.
- The model probably fails because the high value houses can be new houses in cities or old houses in good locations , the price evolution is not a straight line, it depends on many features.

3. Polynomial Regression

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

poly=PolynomialFeatures(degree=2,include_bias=False)
X_train_poly=poly.fit_transform(X_train_scaled)
X_test_poly=poly.transform(X_test_scaled)


In [ ]:
model=LinearRegression()
model.fit(X_train_poly, y_train)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y_pred = model.predict(X_test_poly)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.3f}")

In [ ]:
plt.scatter(y_test, y_pred, alpha=0.3)
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Polynomial Regression: Actual vs Predicted Prices")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.show()

- We can see that the Polynomial regression obtained better results than the linear one because it solves the biggest problem that we have with linear regression: the strict assumption of linearity.
- Mathematically, while Linear Regression fits a straight line (a first-degree polynomial), Polynomial Regression captures non-linear relationships by introducing higher-degree terms (such as $x^2$ or $x^3$). This allows the model to capture complex patterns in the data that a simple straight line would inevitably miss
- We still have a major problem , our model predicts negative prices wich is impossible so we have to solve this by applying a ln to median house value , doing that the model will work with smaller values and after predicting the price in ln scale we will use the Euler's number to convert the value back to currency.

In [ ]:
y_train_log = np.log(y_train)
y_test_log = np.log(y_test)


In [ ]:
model.fit(X_train_poly, y_train_log)

In [ ]:
y_pred_log = model.predict(X_test_poly)
y_pred_real = np.exp(y_pred_log)

mae = mean_absolute_error(y_test, y_pred_real)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_real))
r2 = r2_score(y_test, y_pred_real)

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.3f}")

In [ ]:
plt.scatter(y_test, y_pred_real, alpha=0.3)
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Linear Regression: Actual vs Predicted Prices")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.show()

- Now that I solved the negative values problem we have an ok model that predicts the median house value pretty well even though the datas were not perfect.
- Exactlly like the other 2 models tested , this model works well on normal values but fails on expensive ones.

4. LightGBM

In [ ]:
from sklearn.model_selection import GridSearchCV
from lightgbm import LGBMRegressor
lgb_model_GCV=LGBMRegressor(random_state=42)

param_grid={'n_estimators':[100,200,300],'learning_rate':[0.1,0.2,0.3],'num_leaves':[10,20,31]}
grid_search=GridSearchCV(estimator=lgb_model_GCV,param_grid=param_grid,cv=5,scoring='neg_mean_absolute_error',n_jobs=-1)

grid_search.fit(X_train, y_train)
best_lgb=grid_search.best_estimator_


In [ ]:
y_pred=best_lgb.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.3f}")

In [ ]:
plt.scatter(y_test, y_pred, alpha=0.3)
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Linear Regression: Actual vs Predicted Prices")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.show()

- Based on performances the LightGBM model is the best , the model scored the best performances and it's a fast and light model.
- Because of the mathematical concept behind the model , the LGBM predictions were the best yet, performing good even on the outliers , it still has some issues but it's better than the others


# Final comparison
- The tabel below shows how the tested models performed on the house_pricing dataset

| Model | MAE | RMSE | R² Score |
| :--- | :---: | :---: | :---: |
| LightGBM | 27,123.15 | 40,057.36 | 0.832 |
| Polynomial Regression | 35,539.92 | 53,010.91 | 0.705 |
| Linear Regression | 42,430.49 | 58,585.19 | 0.640 |
| Random Forest Regression | 29,006.15 | 43,506.06 | 0.801 |